In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import os
os.environ["WANDB_DISABLED"] = "true"


In [ ]:
import os
import pandas as pd
import torch
import librosa
from torch.utils.data import Dataset, DataLoader
from transformers import WhisperProcessor, WhisperForConditionalGeneration, Seq2SeqTrainingArguments, Seq2SeqTrainer

BASE = '/kaggle/input/nppe-2-automatic-disfluency-restoration'
TRAINCSV = os.path.join(BASE, 'train.csv')
TESTCSV = os.path.join(BASE, 'test.csv')
AUDIODIR = os.path.join(BASE, 'downloaded_audios')

train_df = pd.read_csv(TRAINCSV)
test_df = pd.read_csv(TESTCSV)

MODELNAME = "collabora/whisper-small-hindi"
processor = WhisperProcessor.from_pretrained(MODELNAME)
model = WhisperForConditionalGeneration.from_pretrained(MODELNAME)
model = model.to('cuda' if torch.cuda.is_available() else 'cpu')

class WhisperDataset(Dataset):
    def __init__(self, df, audiodir, is_test=False):
        self.df = df
        self.audiodir = audiodir
        self.is_test = is_test

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        id_str = str(row['id']).zfill(10)
        audiopath = os.path.join(self.audiodir, f"{id_str}.wav")
        if os.path.exists(audiopath):
            audio, sr = librosa.load(audiopath, sr=16000)
            input_features = processor(audio, sampling_rate=sr, return_tensors="pt").input_features[0]
        else:
            input_features = None
        labels = processor.tokenizer(row['transcript'], return_tensors="pt").input_ids[0] if not self.is_test else torch.tensor([])
        return {"input_features": input_features, "labels": labels, "has_audio": os.path.exists(audiopath), "row_id": row['id']}

def collate_fn(batch):
    batch = [item for item in batch if item["input_features"] is not None]
    if not batch: return None
    input_features = torch.stack([item["input_features"] for item in batch])
    labels = torch.nn.utils.rnn.pad_sequence([item["labels"] for item in batch], batch_first=True, padding_value=-100)
    return {"input_features": input_features, "labels": labels}

train_data = WhisperDataset(train_df, AUDIODIR, is_test=False)

training_args = Seq2SeqTrainingArguments(
    output_dir="./whisper-finetune",
    per_device_train_batch_size=8,
    gradient_accumulation_steps=1,
    fp16=True,
    learning_rate=2e-4,      # Changed LR for speed/accuracy tradeoff
    num_train_epochs=4,      # Bump epochs for better fitting
    logging_steps=40,
    save_steps=201,
    report_to=None,
    predict_with_generate=True
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_data,
    data_collator=collate_fn,
    tokenizer=processor.feature_extractor
)
trainer.train()

# Inference and Submission
test_data = WhisperDataset(test_df, AUDIODIR, is_test=True)
submission = []
model.eval()
for idx in range(len(test_data)):
    d = test_data[idx]
    row_id = d["row_id"]
    if d["has_audio"]:
        inputs = d["input_features"].unsqueeze(0).to(model.device)
        with torch.no_grad():
            pred_ids = model.generate(inputs, max_length=128)
            pred = processor.tokenizer.decode(pred_ids[0], skip_special_tokens=True)
    else:
        pred = ""
    submission.append({"id": row_id, "transcript": pred})

sub_df = pd.DataFrame(submission)
sub_df.to_csv("submission.csv", index=False)


In [ ]:
import os
from tqdm import tqdm

preds = []

for _, row in tqdm(test_df.iterrows(), total=len(test_df)):
    id_str = str(row['id']).zfill(10)
    audiopath = os.path.join(AUDIODIR, f"{id_str}.wav")
    try:
        audio, sr = librosa.load(audiopath, sr=16000)
        # Extract features using processor
        inputs = processor(audio, sampling_rate=16000, return_tensors="pt")
        input_features = inputs.input_features.to(model.device)
        # Inference
        with torch.no_grad():
            pred_ids = model.generate(input_features)
        transcript = processor.decode(pred_ids[0], skip_special_tokens=True)
    except FileNotFoundError:
        # Missing audio: set empty transcript
        transcript = ""
        print(f"Warning: Audio file not found for ID {id_str}. Using empty string.")
    preds.append((row['id'], transcript))

# Create submission
submission = pd.DataFrame(preds, columns=["id", "transcript"])
submission.to_csv("submission.csv", index=False)

# Optional: View first few rows
print(submission.head())



